#### Importações

In [1]:
import pandas as pd
import numpy as np
import requests
from io import BytesIO
import pickle
import os

#### Caminhos e Conexão com Sharepoint

In [2]:
# =============================================================================
# CONFIGURAÇÕES DE CAMINHO (ajuste para o seu ambiente)
# =============================================================================
ONEDRIVE = r"C:\Users\anderson.pereira\OneDrive - Sistema FIEB\Inteligência de Mercado"
PATH_CURSO_TECNICO_NOVO = r"C:\Users\anderson.pereira\OneDrive - Sistema FIEB\Inteligência de Mercado\Projetos\Projeto_Cursos_Técnicos\IM - Inteligência de Mercado\deploy_recomendacao_im\Fonte_Curso_Tecnico_Original\CURSO_TECNICO_ORIGINAL.xlsx"
PATH_OBSERVATORIO   = r"C:\Users\anderson.pereira\OneDrive - Sistema FIEB\Inteligência de Mercado\Observartório_PA\POD_03062026.xlsx"
PATH_POD            = r"C:\Users\anderson.pereira\OneDrive - Sistema FIEB\Inteligência de Mercado\Observartório_PA\extracaoPOD_ed202501_b.xlsx"
PATH_BASE_AJUSTADA  = r"C:\Users\anderson.pereira\OneDrive - Sistema FIEB\Inteligência de Mercado\Projetos\Projeto_Cursos_Técnicos\Base_Ajustes_Manuais_BANCO_DE_DADOS_CHP\Base_Ajustada.xlsx"
PATH_BASE_EVASAO    = r"C:\Users\anderson.pereira\OneDrive - Sistema FIEB\Inteligência de Mercado\Base_Evasão\Evasão"          # pasta com arquivos de evasão
PATH_CONCORRENTES   = r"C:\Users\anderson.pereira\OneDrive - Sistema FIEB\Inteligência de Mercado\Projetos\Projeto_Cursos_Técnicos\IM - Inteligência de Mercado\deploy_recomendacao_im\Fonte_Concorrencia_SISTEC_x_WEB\df_concorrencia.xlsx"
PATH_GEO            = r"C:\Users\anderson.pereira\OneDrive - Sistema FIEB\Inteligência de Mercado\Fontes_NDS\GEO UO MAIS PRÓXIMA.xlsx"

#### dUNIDADE

In [3]:
# =============================================================================
# 1. dUNIDADE
# =============================================================================
def build_dUNIDADE():
    unidade_2 = ["ALAGOINHAS","BARREIRAS","CAMAÇARI","CANDEIAS","EUNÁPOLIS","FEIRA DE SANTANA",
                 "GUANAMBI","ILHÉUS","IPIRÁ","JACOBINA","JEQUIÉ","JUAZEIRO","LAURO DE FREITAS",
                 "LEM","SALVADOR","SALVADOR","SANTO ANTÔNIO DE JESUS","SENHOR DO BONFIM",
                 "SERRINHA","SIMÕES FILHO","TEIXEIRA DE FREITAS","VITÓRIA DA CONQUISTA"]
    unidade   = ["ALAGOINHAS","BARREIRAS","CAMAÇARI","CANDEIAS","EUNÁPOLIS","FEIRA DE SANTANA",
                 "GUANAMBI","ILHÉUS","IPIRÁ","JACOBINA","JEQUIÉ","JUAZEIRO","LAURO DE FREITAS",
                 "LEM","DENDEZEIROS","CIMATEC","SANTO ANTÔNIO DE JESUS","SENHOR DO BONFIM",
                 "SERRINHA","SIMÕES FILHO","TEIXEIRA DE FREITAS","VITÓRIA DA CONQUISTA"]
    regional  = ["CENTRAL","OESTE","METROPOLITANA","","","CENTRAL","","SUL","","NORTE",
                 "SUDOESTE","NORTE","METROPOLITANA","OESTE","SALVADOR","SALVADOR","CENTRAL",
                 "NORTE","CENTRAL","","EXTREMO SUL","SUDOESTE"]
    return pd.DataFrame({"UNIDADE_2": unidade_2, "UNIDADE": unidade, "REGIONAL": regional})

dUNIDADE = build_dUNIDADE()

#### dVAGAS

In [4]:
# =============================================================================
# 12. dVagas  (quarto pasted)
# =============================================================================
def build_dVagas():
    df_raw = pd.read_excel(PATH_CURSO_TECNICO_NOVO, sheet_name="BANCO DE DADOS")
    df = df_raw[df_raw["ANO"] != "Total"].copy()
    df["ANO"]      = pd.to_numeric(df["ANO"], errors="coerce").astype("Int64")
    df["SEMESTRE"] = pd.to_numeric(df["SEMESTRE"], errors="coerce").astype("Int64")
    df["UNIDADE"]  = df["UNIDADE"].str.replace("LUÍS EDUARDO MAGALHÃES","LEM", regex=False)
    df["TURNO"]    = df["TURNO"].str.replace("NOTURNO","NOT",   regex=False).str.replace("VESPERTINO","VESP", regex=False)
    df = df[df["ANO"].isin([2023,2024,2025])]
    df = df.drop(columns=["TURMA","MATRÍCULAS PAGANTE","MATRÍCULAS BOLSISTA","MATRÍCULAS SIND","CANCELADA",
                           "MATRÍCULAS TOTAL","MATRICULA","MENSALIDADE","FORMAS DE PAGAMENTO","CLASSIFICAÇÃO "], errors="ignore")
    df = df.drop_duplicates()

    df["UNIDADE_2"] = df["UNIDADE"].map(lambda u: "SALVADOR" if u in ("CIMATEC","DENDEZEIROS") else u)
    df["REGIONAL"]  = df["REGIONAL"].str.replace("CIMATEC","SALVADOR",    regex=False)
    df["REGIONAL"]  = df["REGIONAL"].str.replace("DENDEZEIROS","SALVADOR", regex=False)

    # duplicar linhas Salvador
    linhas_nulas  = df[df["UNIDADE"].isna() & (df["UNIDADE_2"] == "SALVADOR")].copy()
    lc = linhas_nulas.copy(); lc["UNIDADE"] = "CIMATEC"
    ld = linhas_nulas.copy(); ld["UNIDADE"] = "DENDEZEIROS"
    df = pd.concat([df, lc, ld], ignore_index=True)

    df["UNIDADE"] = df.apply(lambda r: r["UNIDADE_2"] if pd.isna(r["UNIDADE"]) else r["UNIDADE"], axis=1)
    df = df[df["UNIDADE"] != "SALVADOR"]
    df["MODALIDADE"] = df["MODALIDADE"].fillna("-")
    df["TURNO"]      = df["TURNO"].fillna("-")

    # Fallback de vagas: 2025 > 2024 > 2023
    def vagas_fallback(row):
        subset = df[
            (df["SEMESTRE"] == row["SEMESTRE"]) &
            (df["UNIDADE"]   == row["UNIDADE"]) &
            (df["CURSO"]     == row["CURSO"]) &
            (df["MODALIDADE"]== row["MODALIDADE"]) &
            (df["TURNO"]     == row["TURNO"])
        ]
        for ano in [2025, 2024, 2023]:
            v = subset[subset["ANO"] == ano][" VAGAS"]
            if len(v) and not pd.isna(v.max()):
                return v.max()
        return None

    df["VAGAS_2"] = df.apply(vagas_fallback, axis=1)
    df["COD_VAGAS"] = (
        df["SEMESTRE"].astype(str) + df["UNIDADE"] + df["CURSO"] + df["MODALIDADE"] + df["TURNO"]
    )
    df = df.drop(columns=[" VAGAS","ANO"], errors="ignore").drop_duplicates()
    return df

dVagas = build_dVagas()

#### dTURNO | dOFERTA_SENAI | dCLASSIFICACAO | dMODALIDADE | dANO

In [5]:
# =============================================================================
# 13. Dimensões simples
# =============================================================================
dTURNO = pd.DataFrame({"TURNO": ["EAD","MAT","VESP","NOT","-"]})
dOFERTA_SENAI = pd.DataFrame({"Oferta Senai": ["Sim","Não"]})
dCLASSIFICACAO = pd.DataFrame({"CLASSIFICAÇÃO": ["Oportunidade","Risco","Provável Sucesso","Sem avaliação"]})
dMODALIDADE = pd.DataFrame({"MODALIDADE": ["PRESENCIAL","SEMIPRESENCIAL","-"]})
dANO = pd.DataFrame({"ANO": [2023,2024,2025,2026]})

#### GEO_UO_MAIS_PROXIMA

In [6]:
# =============================================================================
# 2. dGEO_UO_AJUSTE_MANUAL
# =============================================================================
dGEO_UO_AJUSTE_MANUAL = pd.DataFrame({
    "MUNICIPIO_ORIGEM":    ["PETROLINA"],
    "MUNICIPIO_UO_PRÓXIMA": ["JUAZEIRO"]
})

# =============================================================================
# 3. GEO UO MAIS PRÓXIMA
# =============================================================================
def build_GEO_UO():
    df = pd.read_excel(PATH_GEO, sheet_name="GEO UO MAIS PRÓXIMA", header=0)
    df = df[["MUNICIPIO_ORIGEM", "MUNICIPIO_UO_PRÓXIMA"]].drop_duplicates()
    df["MUNICIPIO_ORIGEM"]     = df["MUNICIPIO_ORIGEM"].str.upper()
    df["MUNICIPIO_UO_PRÓXIMA"] = df["MUNICIPIO_UO_PRÓXIMA"].str.upper()
    df["MUNICIPIO_ORIGEM"] = df["MUNICIPIO_ORIGEM"].str.replace("IUIU", "IUIÚ", regex=False)
    df["MUNICIPIO_ORIGEM"] = df["MUNICIPIO_ORIGEM"].str.replace("ITAETÊ", "ITAETÉ", regex=False)
    df = pd.concat([df, dGEO_UO_AJUSTE_MANUAL], ignore_index=True)
    return df

GEO_UO_MAIS_PROXIMA = build_GEO_UO()

#### Concorrentes

In [7]:
# =============================================================================
# 11. Base_Concorrentes
# =============================================================================
def build_Base_Concorrentes(GEO_UO_MAIS_PROXIMA):
    df = pd.read_excel(PATH_CONCORRENTES, sheet_name="Sheet1", header=None)
    df.columns = df.iloc[0]; df = df[1:].reset_index(drop=True)

    df = df.merge(GEO_UO_MAIS_PROXIMA, left_on="MUNICÍPIO", right_on="MUNICIPIO_ORIGEM", how="left")
    df = df.rename(columns={"MUNICIPIO_UO_PRÓXIMA":"UNIDADE","CONCORRENTE":"INSTITUIÇÃO"})
    df["QTD"] = 1
    df["UNIDADE"] = df["UNIDADE"].str.replace("LUÍS EDUARDO MAGALHÃES","LEM", regex=False)
    df = df.rename(columns={"QTD":"CONCORRENTES"})
    df["COD_Concorrentes"] = df["UNIDADE"].fillna("") + df["CURSO"].fillna("") + df["MODALIDADE"].fillna("")
    df = df.sort_values("COD_Concorrentes")
    df["CONCORRENTES"] = pd.to_numeric(df["CONCORRENTES"], errors="coerce").astype("Int64")
    return df.reset_index(drop=True)

Base_Concorrentes = build_Base_Concorrentes(GEO_UO_MAIS_PROXIMA)

#### Dados do Observatório

##### Observatorio

In [8]:
# =============================================================================
# 4. Observatorio
# =============================================================================
CURSOS_EXCLUIR = {"Agricultura","Agroecologia","Agropecuária","Florestas","Fruticultura"}

def build_Observatorio():
    df = pd.read_excel(PATH_OBSERVATORIO, sheet_name="Export", header=None)
    # promover cabeçalhos
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    df = df[["Unidade","Catálogo - Curso Técnico","Oferta Senai","Classificação"]].astype(str)
    # remover cursos agrícolas
    df = df[~df["Catálogo - Curso Técnico"].isin(CURSOS_EXCLUIR | {"nan"})]
    # normalizar unidade
    df["Unidade"] = df["Unidade"].str.lower().str.replace(
        "luís eduardo magalhães", "lem", regex=False
    ).str.upper()
    df = df.rename(columns={"Unidade": "UNIDADE_2"})
    df["ANO"] = 2026
    df["COD_Observatorio"] = df["UNIDADE_2"] + df["Catálogo - Curso Técnico"]
    return df

Observatorio = build_Observatorio()

c:\Users\anderson.pereira\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


##### POD_ed20251

In [9]:
# =============================================================================
# 5. POD_ed20251
# =============================================================================
def build_POD_ed20251():
    df = pd.read_excel(PATH_POD, sheet_name="Planilha2")
    df = df[["Curso","Unidade","Modalidade da Oferta","Indicador de Renda para Oferta","Turmas Potenciais"]]
    df["UNIDADE"]  = df["Unidade"].str.lower().str.replace(
        "luís eduardo magalhães", "lem", regex=False
    ).str.upper()
    df["MODALIDADE"] = df["Modalidade da Oferta"].str.upper()
    df["Turmas Potenciais"] = df["Turmas Potenciais"].round(2)
    df = df.rename(columns={
        "Indicador de Renda para Oferta": "Esforço de Venda",
        "Curso": "Curso"
    })
    df["COD_POD"] = df["UNIDADE"] + df["Curso"] + df["MODALIDADE"]
    df = df[["UNIDADE","Curso","MODALIDADE","Esforço de Venda","Turmas Potenciais","COD_POD"]]
    return df

POD_ed20251 = build_POD_ed20251()

##### BD_OBSER_PROV_SUCESS_OPORT

In [10]:
# =============================================================================
# 15. BD_OBSER_PROV_SUCESS_OPORT  (segundo pasted — gerada a partir do BD_Plan)
#     Esta tabela no PQ é derivada de BD_Plan_Curso_Tecnico após joins com
#     Observatorio e POD_ed20251. Reconstruímos aqui essa lógica.
# =============================================================================
def build_BD_OBSER_PROV_SUCESS_OPORT(Observatorio, POD_ed20251):
    df_raw = pd.read_excel(PATH_CURSO_TECNICO_NOVO, sheet_name="BANCO DE DADOS")
    df = df_raw[df_raw["ANO"] != "Total"].copy()
    df["ANO"]      = pd.to_numeric(df["ANO"], errors="coerce").astype("Int64")
    df["SEMESTRE"] = pd.to_numeric(df["SEMESTRE"], errors="coerce").astype("Int64")
    df["UNIDADE"]  = df["UNIDADE"].str.replace("LUÍS EDUARDO MAGALHÃES","LEM", regex=False)
    df["TURNO"]= df["TURNO"].str.upper()
    df["TURNO"]    = df["TURNO"].str.replace("NOTURNO","NOT", regex=False).str.replace("VESPERTINO","VESP", regex=False).str.replace("MATUTINO","MAT", regex=False)
    df = df[(df["ANO"].isin([2023,2024,2025,2026])) & (df["SEMESTRE"] == 1)]

    drop_cols = ["SEMESTRE","MATRÍCULAS TOTAL","MATRICULA","MENSALIDADE","FORMAS DE PAGAMENTO","CLASSIFICAÇÃO "]
    df = df.drop(columns=drop_cols, errors="ignore")
    df = df.rename(columns={"MATRÍCULAS PAGANTE":"PAGANTE","MATRÍCULAS BOLSISTA":"GRATUITO","MATRÍCULAS SIND":"SINDICATO"})
    
    group_keys = ["ANO","REGIONAL","UNIDADE","CURSO","MODALIDADE","TURNO"," VAGAS","TURMA"]
    df_grouped_1 = df.groupby(group_keys, dropna=False, as_index=False).agg(PAGANTE=("PAGANTE","sum"), 
                                                  GRATUITO=("GRATUITO","sum"),
                                                  SINDICATO=("SINDICATO","sum"), 
                                                  CANCELADA=("CANCELADA","sum")).reset_index()
    df_grouped_1 = df_grouped_1.drop_duplicates()

    df_pivot_table = df_grouped_1.melt(id_vars=group_keys, 
                                       value_vars=["PAGANTE","GRATUITO","SINDICATO","CANCELADA"],
                                       var_name="CONDIÇÃO", value_name="Valor")
    df_pivot_table = df_pivot_table[(df_pivot_table['Valor'].notnull()) & 
                                    (df_pivot_table['Valor'] != 0) &
                                    (df_pivot_table['Valor'] != 'nan')]
    df_pivot_table['ANO'] = df_pivot_table['ANO']
    df_pivot_table = df_pivot_table.drop(columns=[" VAGAS", "TURMA", "Valor"], errors="ignore")
    df_pivot_table = df_pivot_table.drop_duplicates()

    df_pivot_table["UNIDADE_2"] = df_pivot_table["UNIDADE"].map(lambda u: "SALVADOR" if u in ("CIMATEC","DENDEZEIROS") else u)
    df_pivot_table["COD_Observatorio"] = df_pivot_table["UNIDADE_2"] + df_pivot_table["CURSO"]

    # join Observatorio
    df_Pivot_Observ = df_pivot_table.merge(
        Observatorio[["COD_Observatorio","Oferta Senai","Classificação"]],
        on="COD_Observatorio", how="left").rename(columns={"Oferta Senai":"OFERTA_SENAI","Classificação":"CLASSIFICACAO"})
    df_Pivot_Observ = df_Pivot_Observ.drop_duplicates()

    # join POD
    df_Pivot_Observ["COD_POD"] = df_Pivot_Observ["UNIDADE_2"] + df_Pivot_Observ["CURSO"] + df_Pivot_Observ["MODALIDADE"].fillna("")
    df_Pivot_Observ_POD = df_Pivot_Observ.merge(POD_ed20251[["COD_POD","Esforço de Venda"]], on="COD_POD", how="left")
    df_Pivot_Observ_POD["Esforço de Venda"] = df_Pivot_Observ_POD["Esforço de Venda"].fillna("-")
    df_Pivot_Observ_POD = df_Pivot_Observ_POD.drop_duplicates()

    df_Pivot_Observ_POD["REGIONAL"] = df_Pivot_Observ_POD["REGIONAL"].str.replace("CIMATEC","SALVADOR",    regex=False)
    df_Pivot_Observ_POD["REGIONAL"] = df_Pivot_Observ_POD["REGIONAL"].str.replace("DENDEZEIROS","SALVADOR", regex=False)

    df_Pivot_Observ_POD["COD_CLASS_OPORT_SUCESS"] = (df_Pivot_Observ_POD["COD_Observatorio"].fillna("") +df_Pivot_Observ_POD["OFERTA_SENAI"].fillna("") +df_Pivot_Observ_POD["CLASSIFICACAO"].fillna(""))
    df_Pivot_Observ_POD = df_Pivot_Observ_POD.drop_duplicates().reset_index(drop=True)

    return df_Pivot_Observ_POD 

BD_OBSER_PROV_SUCESS_OPORT = build_BD_OBSER_PROV_SUCESS_OPORT(Observatorio, POD_ed20251)

##### Observatorio_2

In [11]:
# =============================================================================
# 8. Observatorio_2
# =============================================================================
def build_Observatorio_2(Observatorio, dUNIDADE, BD_OBSER_PROV_SUCESS_OPORT):
    df = Observatorio.copy()
    df["MODALIDADE"] = np.nan
    df["TURNO"]      = np.nan
    df["CONDIÇÃO"]   = np.nan

    # join com dUNIDADE para obter REGIONAL
    df_dUNID = df.merge(dUNIDADE[["UNIDADE_2","REGIONAL"]], on="UNIDADE_2", how="left")

    df_dUNID[" VAGAS"] = np.nan
    df_dUNID["TURMA"]  = 0
    df_dUNID["Valor"]  = 0
    df_dUNID["SEMESTRE"] = 1

    df_dUNID = df_dUNID.rename(columns={"Catálogo - Curso Técnico": "CURSO"})
    df_dUNID["UNIDADE"] = np.nan

    col_order = ["ANO","SEMESTRE","UNIDADE","CURSO","MODALIDADE","TURNO","CONDIÇÃO",
                 "REGIONAL"," VAGAS","TURMA","Valor","UNIDADE_2","COD_Observatorio",
                 "Oferta Senai","Classificação"]
    # garantir colunas existentes
    for c in col_order:
        if c not in df_dUNID.columns:
            df_dUNID[c] = np.nan
    df_dUNID = df_dUNID[col_order]
    df_dUNID = df_dUNID.rename(columns={"Oferta Senai":"OFERTA_SENAI","Classificação":"CLASSIFICACAO"})

    # filtrar fora Risco
    df_dUNID = df_dUNID[df_dUNID["CLASSIFICACAO"] != "Risco"]

    # # criar chave
    df_dUNID["COD_CLASS_OPORT_SUCESS"] = (df_dUNID["COD_Observatorio"].fillna("") + df_dUNID["OFERTA_SENAI"].fillna("") + df_dUNID["CLASSIFICACAO"].fillna(""))

    # anti-join com BD_OBSER_PROV_SUCESS_OPORT
    existing = set(BD_OBSER_PROV_SUCESS_OPORT["COD_CLASS_OPORT_SUCESS"].dropna())
    df_dUNID_BD_OBSER_PROV = df_dUNID[~df_dUNID["COD_CLASS_OPORT_SUCESS"].isin(existing)]
    df_dUNID_BD_OBSER_PROV = df_dUNID_BD_OBSER_PROV.sort_values("COD_Observatorio")

    # remover colunas auxiliares e reordenar
    df_dUNID_BD_OBSER_PROV = df_dUNID_BD_OBSER_PROV.drop(columns=["COD_CLASS_OPORT_SUCESS","COD_Observatorio","OFERTA_SENAI","CLASSIFICACAO"])
    col_final = ["ANO","SEMESTRE","REGIONAL","UNIDADE","CURSO","MODALIDADE","TURNO"," VAGAS", "TURMA","CONDIÇÃO","Valor","UNIDADE_2"]
    
    for c in col_final:
        if c not in df_dUNID_BD_OBSER_PROV.columns:
            df_dUNID_BD_OBSER_PROV[c] = np.nan
    df_dUNID_BD_OBSER_PROV = df_dUNID_BD_OBSER_PROV[col_final]

    # duplicar para semestre 2
    sem2 = df_dUNID_BD_OBSER_PROV.copy()
    sem2["SEMESTRE"] = 2
    df_final = pd.concat([df_dUNID_BD_OBSER_PROV, sem2], ignore_index=True)
    df_final.drop_duplicates(inplace=True)
    return df_final.reset_index(drop=True)

Observatorio_2 = build_Observatorio_2(Observatorio, dUNIDADE, BD_OBSER_PROV_SUCESS_OPORT)

#### build_base_ajustada

In [12]:
# =============================================================================
# 6. base_ajustada
# =============================================================================
def build_base_ajustada():
    df = pd.read_excel(PATH_BASE_AJUSTADA, sheet_name="Planilha1")
    df["ANO"]      = df["ANO"].astype("Int64")
    df["SEMESTRE"] = df["SEMESTRE"].astype("Int64")
    return df

base_ajustada = build_base_ajustada()

#### BD_Plan_Curso_Tecnico

In [13]:
# =============================================================================
# 9. BD_Plan_Curso_Tecnico  (primeiro pasted)
# =============================================================================
def build_BD_Plan_Curso_Tecnico(Observatorio_2, base_ajustada):
    # --- Bloco 1: Fonte ---
    df_raw = df = pd.read_excel(PATH_CURSO_TECNICO_NOVO, sheet_name="BANCO DE DADOS")

    # --- Bloco 2: Processar tabela de fatos ---
    df = df_raw[df_raw["ANO"] != "Total"].copy(deep=True)
    df["ANO"]      = df["ANO"].astype(int)
    df["SEMESTRE"] = df["SEMESTRE"].astype(int)
    df = df[(df["ANO"] >= 2023) & (df["ANO"] <= 2026)]
    df = df.drop(columns=["MATRÍCULAS TOTAL","MATRICULA","MENSALIDADE","FORMAS DE PAGAMENTO","CLASSIFICAÇÃO "], errors="ignore")

    df["UNIDADE"] = df["UNIDADE"].str.replace("LUÍS EDUARDO MAGALHÃES","LEM", regex=False)
    df["TURNO"] = df["TURNO"].str.upper()
    df["TURNO"]   = df["TURNO"].str.replace("NOTURNO","NOT",  regex=False).str.replace("VESPERTINO","VESP", regex=False).str.replace("MATUTINO","MAT", regex=False)

    int_cols = [" VAGAS","TURMA","MATRÍCULAS PAGANTE","MATRÍCULAS BOLSISTA","CANCELADA","MATRÍCULAS SIND"]
    for c in int_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    df = df.rename(columns={ "MATRÍCULAS PAGANTE": "PAGANTE", 
                            "MATRÍCULAS BOLSISTA": "GRATUITO", 
                            "MATRÍCULAS SIND": "SINDICATO"})

    group_keys = ["ANO","SEMESTRE","REGIONAL","UNIDADE","CURSO","MODALIDADE","TURNO"," VAGAS","TURMA"]
    df_group = df.groupby(group_keys, dropna=False, as_index=False).agg(PAGANTE=("PAGANTE","sum"), 
                                                        GRATUITO=("GRATUITO","sum"),
                                                        SINDICATO=("SINDICATO","sum"), 
                                                        CANCELADA=("CANCELADA","sum")).reset_index()

    df_group_pivot = df_group.melt(id_vars=group_keys, 
                                   value_vars=["PAGANTE","GRATUITO","SINDICATO","CANCELADA"],
                                   var_name="CONDIÇÃO", 
                                   value_name="Valor")
    # df_group_pivot = df_group_pivot[(df_group_pivot['Valor'].notnull()) & 
    #                                 (df_group_pivot['Valor'] != 0) &
    #                                 (df_group_pivot['Valor'] != 'nan')]
    df_group_pivot = df_group_pivot.drop_duplicates().reset_index(drop=True)
    df_group_pivot["UNIDADE_2"] = df_group_pivot["UNIDADE"].map(lambda u: "SALVADOR" if u in ("CIMATEC","DENDEZEIROS") else u)

    # # --- Bloco 3: Anexar Observatorio_2 e duplicar linhas Salvador ---
    # Observatorio_2 precisa ter as mesmas colunas; completar colunas faltantes
    obs2 = Observatorio_2.copy()
    for c in df_group_pivot.columns:
        if c not in obs2.columns:
            obs2[c] = np.nan
    df_obs2 = pd.concat([df_group_pivot, obs2[df_group_pivot.columns]], ignore_index=True).reset_index(drop=True)

    linhas_principais = df_obs2[~(df_obs2['UNIDADE'].notnull() & (df_obs2["UNIDADE_2"] == "SALVADOR"))]
    linhas_molde =  df_obs2[(df_obs2['UNIDADE'].isnull()) & (df_obs2["UNIDADE_2"] == "SALVADOR")]

    linhas_cimatec = linhas_molde.copy(deep=True); linhas_cimatec["UNIDADE"] = "CIMATEC"
    linhas_dendezeiros_0 = linhas_molde.copy(deep=True); linhas_dendezeiros_0["UNIDADE"] = "DENDEZEIROS"
    linhas_dendezeiros_1 = linhas_dendezeiros_0[linhas_dendezeiros_0["CURSO"] != "Rádio e Televisão"]

    df_obs2_SSA = pd.concat([linhas_principais, linhas_cimatec, linhas_dendezeiros_1], ignore_index=True)

    # consolidar UNIDADE
    df_obs2_SSA["UNIDADE"] = df_obs2_SSA.apply(lambda r: r["UNIDADE_2"] if pd.isna(r["UNIDADE"]) else r["UNIDADE"], axis=1)

    # preencher nulos em MODALIDADE e TURNO
    df_obs2_SSA["MODALIDADE"] = df_obs2_SSA["MODALIDADE"].fillna("-")
    df_obs2_SSA["TURNO"] = df_obs2_SSA["TURNO"].fillna("-")

    # --- Anexar base_ajustada ---
    for c in df_obs2_SSA.columns:
        if c not in base_ajustada.columns:
            base_ajustada[c] = np.nan
    df_obs2_SSA_bAjust = pd.concat([df_obs2_SSA, base_ajustada[df_obs2_SSA.columns]], ignore_index=True)

    # tipos finais
    df_obs2_SSA_bAjust["Valor"] = pd.to_numeric(df_obs2_SSA_bAjust["Valor"], errors="coerce").astype("Int64")
    df_obs2_SSA_bAjust["ANO"]   = pd.to_numeric(df_obs2_SSA_bAjust["ANO"],   errors="coerce").astype("Int64")

    # --- Bloco 5: Surrogate Keys ---
    u2   = df_obs2_SSA_bAjust["UNIDADE_2"].fillna("")
    u    = df_obs2_SSA_bAjust["UNIDADE"].fillna("")
    cur  = df_obs2_SSA_bAjust["CURSO"].fillna("")
    mod  = df_obs2_SSA_bAjust["MODALIDADE"].fillna("")
    tur  = df_obs2_SSA_bAjust["TURNO"].fillna("")
    cond = df_obs2_SSA_bAjust["CONDIÇÃO"].fillna("")
    ano  = df_obs2_SSA_bAjust["ANO"].astype(str)
    sem  = df_obs2_SSA_bAjust["SEMESTRE"].astype(str)

    df_obs2_SSA_bAjust["COD_Observatorio"]   = u2 + cur
    df_obs2_SSA_bAjust["COD_POD"]            = u2 + cur + mod
    df_obs2_SSA_bAjust["COD_Concorrentes"]   = u2 + cur + mod
    df_obs2_SSA_bAjust["COD_VAGAS"]          = sem + u + cur + mod + tur
    df_obs2_SSA_bAjust["COD_VagasBolsistas"] = ano + u + cur + mod + tur + cond
    df_obs2_SSA_bAjust["COD_IndicaSENAI"]    = ano + u + cur
    df_obs2_SSA_bAjust["COD_Predicoes"]      = ano + sem + u + cur + mod + tur
    df_obs2_SSA_bAjust["COD_Meta"]           = sem + u + cur + mod + tur
    df_obs2_SSA_bAjust["COD_Base_de_Evasao"]  = ano + sem + u + cur + mod + tur + cond

    # # filtrar agregadora SALVADOR
    df_final = df_obs2_SSA_bAjust[df_obs2_SSA_bAjust["UNIDADE"] != "SALVADOR"]

    return df_final

BD_Plan_Curso_Tecnico = build_BD_Plan_Curso_Tecnico(Observatorio_2, base_ajustada)

C:\Users\anderson.pereira\AppData\Local\Temp\ipykernel_37196\2037207824.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_obs2 = pd.concat([df_group_pivot, obs2[df_group_pivot.columns]], ignore_index=True).reset_index(drop=True)
C:\Users\anderson.pereira\AppData\Local\Temp\ipykernel_37196\2037207824.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_obs2_SSA_bAjust = pd.concat([df_obs2_SSA, base_ajustada[df_obs2_SSA.columns]], ignore_index=True)


In [14]:
# print("Distinto - ANO: ",len(BD_Plan_Curso_Tecnico.ANO.value_counts()))
# print("Distinto - UNIDADE: ",len(BD_Plan_Curso_Tecnico.UNIDADE.value_counts()))
# print("Distinto - CURSO: ",len(BD_Plan_Curso_Tecnico.CURSO.value_counts()))
# print("Distinto - MODALIDADE: ",len(BD_Plan_Curso_Tecnico.MODALIDADE.value_counts()))
# print("Distinto - TURNO: ",len(BD_Plan_Curso_Tecnico.TURNO.value_counts()))
# # print("Distinto - CONDIÇÃO: ",len(BD_Plan_Curso_Tecnico.CONDIÇÃO.value_counts()))
# print("Distinto - REGIONAL: ",len(BD_Plan_Curso_Tecnico.REGIONAL.value_counts()))

# checagem = BD_Plan_Curso_Tecnico.sort_values(['ANO','UNIDADE','CURSO'], ascending=True).reset_index(drop=True)
# # FILTRO = checagem[(checagem['Valor'] != 0)] #& (checagem['CURSO'] == 'Administração')].reset_index(drop=True)
# # checagem
# checagem


#### Evasão

In [15]:
# =============================================================================
# 10. Base_de_Evasao  (terceiro pasted — evasão)
# =============================================================================
def build_Base_de_Evasao():
    import glob, os
    arquivos = glob.glob(os.path.join(PATH_BASE_EVASAO, "*.xlsx"))
    frames = []
    for arq in arquivos:
        nome = os.path.basename(arq)
        condicao = "GRATUITO" if "Bolsista" in nome else "PAGANTE"
        tmp = pd.read_excel(arq)
        tmp["CONDIÇÃO"] = condicao
        tmp["_origem"]  = nome
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)

    # primeiros 3 chars do turno
    df["TURNO_3"] = df["TURNO"].str[:3]
    df = df.drop(columns=["EVASÃO","TURNO"], errors="ignore")
    df = df[df["UNIDADE"].notna()]
    df = df[df.apply(lambda r: not all(v in ("", None) for v in r.values), axis=1)]

    df["CURSO"]      = df["CURSO"].str.replace("Técnico em ","", regex=False)
    df["UNIDADE"]    = df["UNIDADE"].str.replace("FEIRA","FEIRA DE SANTANA", regex=False)
    df["UNIDADE"]    = df["UNIDADE"].str.replace("CONQUISTA","VITÓRIA DA CONQUISTA", regex=False)
    df["TIPO CURSO"] = df["TIPO CURSO"].str.replace("DISTÂNCIA","SEMIPRESENCIAL", regex=False)

    df["EVASÃO"]            = df["MATRÍCULAS INICIAL"] - df["MATRICULAS FINAL"]
    df["MATRÍCULAS INICIAL"] = pd.to_numeric(df["MATRÍCULAS INICIAL"], errors="coerce").astype("Int64")
    df["MATRICULAS FINAL"]  = pd.to_numeric(df["MATRICULAS FINAL"],  errors="coerce").astype("Int64")
    df["EVASÃO"]            = pd.to_numeric(df["EVASÃO"],            errors="coerce").astype("Int64")

    df["SEMESTRE"] = df["INÍCIO DA TURMA"].astype(str).str[-1:].astype(int)
    df = df.rename(columns={"INÍCIO DA TURMA":"ANO"})
    df["ANO"] = df["ANO"].astype(str).str[:4].astype(int)

    group_keys = ["ANO","UNIDADE","CURSO","TIPO CURSO","TURNO_3","CONDIÇÃO","SEMESTRE"]
    df = df.groupby(group_keys, dropna=False).agg(
        EVASAO=("EVASÃO","sum")
    ).reset_index()

    df["ANO"] = df["ANO"].astype(str)
    df["COD_Base_de_Evasao"] = (
        df["ANO"] +
        df["SEMESTRE"].astype(str) +
        df["UNIDADE"] + df["CURSO"] + df["TIPO CURSO"] + df["TURNO_3"] + df["CONDIÇÃO"]
    )
    df["ANO"]      = df["ANO"].astype(int)
    df["SEMESTRE"] = df["SEMESTRE"].astype(int)
    return df

Base_de_Evasao = build_Base_de_Evasao()

#### dCURSOS

In [16]:
# =============================================================================
# 14. dCURSOS
# =============================================================================
def build_dCURSOS(BD_Plan_Curso_Tecnico, Observatorio):
    c1 = BD_Plan_Curso_Tecnico[["CURSO"]].drop_duplicates().rename(columns={"CURSO":"Nome do Curso"})
    c2 = Observatorio[["Catálogo - Curso Técnico"]].drop_duplicates().rename(columns={"Catálogo - Curso Técnico":"Nome do Curso"})
    df = pd.concat([c1, c2], ignore_index=True)
    df = df[df["Nome do Curso"].notna() & (df["Nome do Curso"] != "")].drop_duplicates()
    df = df.sort_values("Nome do Curso").reset_index(drop=True)
    df.insert(0, "ID_Curso", range(1, len(df)+1))
    df = df.rename(columns={"Nome do Curso":"CURSO"})
    return df

dCURSOS = build_dCURSOS(BD_Plan_Curso_Tecnico, Observatorio)

#### Salvando arquivos

In [17]:
#======= DIMENSÃO
with open("dUNIDADE.pkl", "wb") as f:
    pickle.dump(dUNIDADE, f)
with open("dVagas.pkl", "wb") as f:
    pickle.dump(dVagas, f)
with open("dTURNO.pkl", "wb") as f:
    pickle.dump(dTURNO, f)
with open("dOFERTA_SENAI.pkl", "wb") as f:
    pickle.dump(dOFERTA_SENAI, f)
with open("dCLASSIFICACAO.pkl", "wb") as f:
    pickle.dump(dCLASSIFICACAO, f)
with open("dMODALIDADE.pkl", "wb") as f:
    pickle.dump(dMODALIDADE, f)
with open("dANO.pkl", "wb") as f:
    pickle.dump(dANO, f)
with open("dCURSOS.pkl", "wb") as f:
    pickle.dump(dCURSOS, f)

#======= FATO
with open("GEO_UO_MAIS_PROXIMA.pkl", "wb") as f:
    pickle.dump(GEO_UO_MAIS_PROXIMA, f)
with open("Base_Concorrentes.pkl", "wb") as f:
    pickle.dump(Base_Concorrentes, f)
with open("Observatorio_2.pkl", "wb") as f:
    pickle.dump(Observatorio_2, f)
with open("base_ajustada.pkl", "wb") as f:
    pickle.dump(base_ajustada, f)
with open("BD_Plan_Curso_Tecnico.pkl", "wb") as f:
    pickle.dump(BD_Plan_Curso_Tecnico, f)
with open("Base_de_Evasao.pkl", "wb") as f:
    pickle.dump(Base_de_Evasao, f)

print("Arquivos salvos com sucesso!")

Arquivos salvos com sucesso!
